In [2]:
import duckdb

In [9]:
conn = duckdb.connect('social_media_for_professionals.duckdb')



# Count the number of tables
tables = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]
print(f"Number of tables: {len(tables)}")

# Count the number of rows in each table
for table in tables:
    row_count = conn.execute(f"SELECT COUNT(*) FROM \"{table}\"").fetchone()[0]
    print(f"Table '{table}' has {row_count} rows.")



Number of tables: 7
Table 'Comments' has 250 rows.
Table 'Connections' has 250 rows.
Table 'GroupMembers' has 250 rows.
Table 'Groups' has 50 rows.
Table 'Messages' has 250 rows.
Table 'Posts' has 250 rows.
Table 'Users' has 250 rows.


In [7]:
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize
fake = Faker()

# Drop all tables if they exist
# Drop dependent tables first to avoid CatalogException due to foreign key constraints
conn.execute("DROP TABLE IF EXISTS GroupMembers")
conn.execute("DROP TABLE IF EXISTS Comments")
conn.execute("DROP TABLE IF EXISTS Connections")
conn.execute("DROP TABLE IF EXISTS Groups")
conn.execute("DROP TABLE IF EXISTS Messages")
conn.execute("DROP TABLE IF EXISTS Posts")
conn.execute("DROP TABLE IF EXISTS Users")

# Recreate tables
conn.execute("""
CREATE TABLE Users (
    user_id INTEGER PRIMARY KEY,
    username VARCHAR NOT NULL UNIQUE,
    email VARCHAR NOT NULL UNIQUE,
    password VARCHAR NOT NULL,
    first_name VARCHAR,
    last_name VARCHAR,
    profile_picture VARCHAR,
    bio VARCHAR,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.execute("""
CREATE TABLE Groups (
    group_id INTEGER PRIMARY KEY,
    group_name VARCHAR NOT NULL,
    description VARCHAR,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.execute("""
CREATE TABLE Posts (
    post_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    content VARCHAR NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.execute("""
CREATE TABLE Comments (
    comment_id INTEGER PRIMARY KEY,
    post_id INTEGER,
    user_id INTEGER,
    content VARCHAR NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.execute("""
CREATE TABLE Messages (
    message_id INTEGER PRIMARY KEY,
    sender_id INTEGER,
    receiver_id INTEGER,
    content VARCHAR NOT NULL,
    sent_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.execute("""
CREATE TABLE Connections (
    connection_id INTEGER PRIMARY KEY,
    user_id_1 INTEGER,
    user_id_2 INTEGER,
    connection_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

# Insert 250 users
insert_user_query = "INSERT INTO Users VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)"
for i in range(1, 251):
    username = fake.unique.user_name()
    email = fake.email()
    password = fake.password()
    first_name = fake.first_name()
    last_name = fake.last_name()
    profile_picture = fake.image_url()
    bio = fake.text(max_nb_chars=200)
    created_at = datetime.now().isoformat()
    conn.execute(insert_user_query, (i, username, email, password, first_name, last_name, profile_picture, bio, created_at))

# Insert 50 groups
insert_group_query = "INSERT INTO Groups VALUES (?, ?, ?, ?)"
for i in range(1, 51):
    group_name = fake.word() + " Group"
    description = fake.sentence()
    created_at = datetime.now().isoformat()
    conn.execute(insert_group_query, (i, group_name, description, created_at))

# Insert 250 posts
insert_post_query = "INSERT INTO Posts VALUES (?, ?, ?, ?)"
for i in range(1, 251):
    user_id = random.randint(1, 250)
    content = fake.text(max_nb_chars=500)
    created_at = datetime.now().isoformat()
    conn.execute(insert_post_query, (i, user_id, content, created_at))

# Insert 250 comments
insert_comment_query = "INSERT INTO Comments VALUES (?, ?, ?, ?, ?)"
for i in range(1, 251):
    post_id = random.randint(1, 250)
    user_id = random.randint(1, 250)
    content = fake.text(max_nb_chars=200)
    created_at = datetime.now().isoformat()
    conn.execute(insert_comment_query, (i, post_id, user_id, content, created_at))

# Insert 250 messages
insert_message_query = "INSERT INTO Messages VALUES (?, ?, ?, ?, ?)"
for i in range(1, 251):
    sender_id = random.randint(1, 250)
    receiver_id = random.randint(1, 250)
    content = fake.text(max_nb_chars=200)
    sent_at = datetime.now().isoformat()
    conn.execute(insert_message_query, (i, sender_id, receiver_id, content, sent_at))

# Insert 250 connections
insert_connection_query = "INSERT INTO Connections VALUES (?, ?, ?, ?)"
for i in range(1, 251):
    user_id_1 = random.randint(1, 250)
    user_id_2 = random.randint(1, 250)
    connection_date = datetime.now().isoformat()
    conn.execute(insert_connection_query, (i, user_id_1, user_id_2, connection_date))

# Create the GroupMembers table first, then insert the first group member and the rest
conn.execute("CREATE TABLE GroupMembers (group_member_id INTEGER, group_id INTEGER, user_id INTEGER, joined_at TIMESTAMP)")
insert_group_member_query = "INSERT INTO GroupMembers VALUES (?, ?, ?, ?)"
conn.execute(insert_group_member_query, (1, 1, 1, datetime.now().isoformat()))
for i in range(2, 251):
    group_id = random.randint(1, 50)
    user_id = random.randint(1, 250)
    joined_at = datetime.now().isoformat()
    conn.execute(insert_group_member_query, (i, group_id, user_id, joined_at))

# COMMIT; add commit bro
conn.commit()

In [8]:
conn.close()

In [ ]:
# This cell can be used for testing queries or inspecting the generated schema and data.

import duckdb

# Connect to the generated DuckDB file


# Example: List all tables
def list_tables(conn):
    result = conn.execute("SHOW TABLES").fetchall()
    print("Tables in database:")
    for row in result:
        print(row[0])

list_tables(conn)

# Example: Show first 5 users
try:
    users = conn.execute("SELECT * FROM Users LIMIT 5").fetchall()
    print("\nFirst 5 users:")
    for user in users:
        print(user)
except Exception as e:
    print("Error querying Users table:", e)

# Example: Count posts
try:
    post_count = conn.execute("SELECT COUNT(*) FROM Posts").fetchone()[0]
    print(f"\nTotal posts: {post_count}")
except Exception as e:
    print("Error querying Posts table:", e)

# Close connection
# conn.close()

